# Intermezzo · L'attrezzeria

La nostra rete scritta in PyTorch, e il gradiente come trasformazione in JAX. Servono `torch` e `jax`.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def sigmoide(z):
    return 1 / (1 + math.exp(-z))

## PyTorch: la rete 3 → 2 → 1

Il kata chiede di provare tre learning rate partendo sempre dagli stessi pesi: per questo il seme si fissa prima di creare la rete.

In [2]:
import torch
import torch.nn as nn

def allena(lr, passi=100, seme=0):
    torch.manual_seed(seme)
    rete = nn.Sequential(nn.Linear(3, 2), nn.ReLU(), nn.Linear(2, 1))
    x = torch.tensor([[0.8, 1.0, 1.0]])
    y = torch.tensor([[0.0]])
    errore = nn.BCEWithLogitsLoss()
    ottimizzatore = torch.optim.SGD(rete.parameters(), lr=lr)
    prima = torch.sigmoid(rete(x)).item()
    for _ in range(passi):
        loss = errore(rete(x), y)
        ottimizzatore.zero_grad()
        loss.backward()
        ottimizzatore.step()
    return prima, torch.sigmoid(rete(x)).item()

for lr in [0.01, 0.1, 1]:
    prima, dopo = allena(lr)
    print(f"learning rate {lr}: da {prima:.0%} a {dopo:.1%}")

learning rate 0.01: da 45% a 35.2%
learning rate 0.1: da 45% a 9.7%
learning rate 1: da 45% a 1.0%


Prova altri semi: con alcune partenze sfortunate entrambi i neuroni nascosti restano spenti dalla ReLU, e si muove solo il bias d'uscita. È il neurone «morto» della soluzione.

In [3]:
for seme in range(5):
    print(f"seme {seme}: " + ", ".join(f"lr {lr} → {allena(lr, seme=seme)[1]:.1%}" for lr in [0.01, 0.1, 1]))

seme 0: lr 0.01 → 35.2%, lr 0.1 → 9.7%, lr 1 → 1.0%


seme 1: lr 0.01 → 39.0%, lr 0.1 → 1.3%, lr 1 → 0.1%
seme 2: lr 0.01 → 40.8%, lr 0.1 → 10.3%, lr 1 → 1.0%


seme 3: lr 0.01 → 37.8%, lr 0.1 → 10.0%, lr 1 → 1.0%


seme 4: lr 0.01 → 34.4%, lr 0.1 → 9.6%, lr 1 → 1.0%


## JAX: il gradiente come trasformazione

In [4]:
import jax
import jax.numpy as jnp

def errore(w, x, y):
    p = jax.nn.sigmoid(jnp.dot(w, x))
    return -(y * jnp.log(p) + (1 - y) * jnp.log(1 - p))

colpe = jax.grad(errore)
g = colpe(jnp.array([2.0, -1.5, 0.1]), jnp.array([0.8, 1.0, 1.0]), 0.0)
for nome, val, atteso in zip(["energia", "pioggia", "compagno"], g, [0.44, 0.55, 0.55]):
    controlla(f"colpa {nome}", float(val), atteso)

✓  colpa energia: calcolato 0.4399, nel libro 0.44
✓  colpa pioggia: calcolato 0.5498, nel libro 0.55
✓  colpa compagno: calcolato 0.5498, nel libro 0.55


**Esercizio 2.** Il bias come quarto peso, con un input che vale sempre 1.

In [5]:
w4 = jnp.array([2.0, -1.5, 0.1, -0.5])
x4 = jnp.array([0.8, 1.0, 1.0, 1.0])
g4 = colpe(w4, x4, 0.0)
p = float(jax.nn.sigmoid(jnp.dot(w4, x4)))
controlla("probabilità", p, 0.43)
controlla("colpa del bias", float(g4[3]), 0.43)
assert abs(float(g4[3]) - (p - 0)) < 1e-6
print("✓  la colpa del bias è uguale al segnale d'errore p − y")

✓  probabilità: calcolato 0.4256, nel libro 0.43
✓  colpa del bias: calcolato 0.4256, nel libro 0.43
✓  la colpa del bias è uguale al segnale d'errore p − y
